<br>
<font>
<div dir=ltr align=center>
<img src="https://cdn.freebiesupply.com/logos/large/2x/sharif-logo-png-transparent.png" width=150 height=150> <br>
<font color=0F5298 size=7>
    Machine learning <br>
<font color=2565AE size=5>
    Computer Engineering Department <br>
    Fall 2025<br>
<font color=3C99D size=5>
    Practical Assignment 5 - NLP - Word Embeddings & Fine-Tuning <br>
</div>
<div dir=ltr align=center>
<font color=0CBCDF size=4>
     Emad Rostamian, Seyed Moein Ayyoubzadeh
<br>
<font color=0CBCDF size=4>
</div>

____

<font color=9999FF size=4>
Full Name : Faraz Doagooye Tehrani
<br>
<font color=9999FF size=4>
Student Number : 402105998

<font color=0080FF size=3>
This notebook covers two related topics in modern NLP. In the first part, we study word embedding methods, including traditional and neural approaches, to understand how semantic information is represented in vector space. In the second part, we explore fine-tuning strategies for pretrained language models, with a focus on parameter-efficient methods such as LoRA, and analyze their behavior during training.
</font>

# Section 1: Exploring Word Embeddings

## 🎯 Objective

In this section, you will **build, explore, and compare different types of word representations (embeddings)** to understand how machines capture the meaning of words from text.


---

## 📚 Dataset

We will work with the complete **_Harry Potter_ series (Books 1–7)**, loaded from a public Hugging Face dataset.

⚠️ **Note:** This text is used **only for educational purposes**.


---

## 🛠️ What You Will Do

You will:
1. **Implement from scratch**
   - TF-IDF
   - Word2Vec-style embeddings (CBOW and Skip-Gram)
   - Nearest-neighbor search (word similarity)
2. **GloVe: Global Vectors for Word Representation**
   - Exploring word vectors
   - Word Analogies

## Part 1: Implement from scratch

## 1. Traditional Word Representations: TF-IDF

Before learning dense vector embeddings, we start with a **traditional and interpretable** way of representing words: **TF-IDF**.

TF-IDF does *not* try to learn meaning from prediction tasks. Instead, it represents words based on how **frequently** they appear and how **informative** they are across documents.

---

## Term Frequency (TF)

Term Frequency measures how often a word appears in a document.

For a word $ w $ in document $ d $:

$$
\text{TF}(w, d) = \frac{\text{count of } w \text{ in } d}{\text{total number of words in } d}
$$

Intuition:
- Words that appear often in a document are likely important *for that document*

---

## Inverse Document Frequency (IDF)

Inverse Document Frequency downweights words that appear in *many* documents.

$$
\text{IDF}(w) = \log\left(\frac{1 + N}{1 + \text{DF}(w)}\right) + 1
$$

Where:
- $ N $ = total number of documents  
- $ \text{DF}(w) $ = number of documents containing word \( w \)

Why the **+1** terms:
- `+1` in the denominator prevents division by zero  
- `+1` in the numerator keeps the ratio ≥ 1  
- `+1` outside the log prevents zero IDF values  

Intuition:
- Common words get low weight  
- Rare words get high weight  
- All words keep a positive contribution  


---

## TF-IDF Score

The final TF-IDF representation combines both ideas:

$$
\text{TF-IDF}(w, d) = \text{TF}(w, d) \times \text{IDF}(w)
$$

Each document is represented as a **high-dimensional sparse vector**, where:
- Each dimension corresponds to a word in the vocabulary
- Values reflect word importance


---

## Your Task

In this section, you will:

1. Load **Harry Potter and the Sorcerer’s Stone**
2. Split the book into **chapters** (each chapter = one document)
3. Build **TF-IDF vectors from scratch**
4. Use TF-IDF to:
   - Find the most important words in a chapter
   - Compute similarity between chapters
   - Compare words based on their TF-IDF representations

Do **not** use `sklearn`’s `TfidfVectorizer`.

The goal is to understand *how TF-IDF works internally*.


In [12]:
# Standard library
import re
import math
import random
from collections import Counter, defaultdict

# Third-party libraries
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Hugging Face
from datasets import load_dataset

# Scikit-learn
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [13]:
# Load dataset
hp_ds = load_dataset("elricwan/HarryPotter")

# Each book is one document
books = hp_ds["train"]

# Extract book texts
documents = [book["content"] for book in books]
documents = documents[:-1]
print(f"Number of books: {len(documents)}")


Number of books: 7


In [14]:
# TODO: Preprocessing : Normalize text and remove punctuation and stopwords
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = text.split()

    tokens = [w for w in tokens if w not in ENGLISH_STOP_WORDS]

    return tokens

docs = [preprocess(doc) for doc in documents]

# TODO: Print number of Tokens in each book(doc)
for i, doc in enumerate(docs):
    print(f"Book {i+1} total tokens: {len(doc)}")



Book 1 total tokens: 38759
Book 2 total tokens: 44838
Book 3 total tokens: 55465
Book 4 total tokens: 95330
Book 5 total tokens: 126688
Book 6 total tokens: 82121
Book 7 total tokens: 94453


In [15]:
# TODO: Build a shared corpus and initial vocabulary
corpus = [token for doc in docs for token in doc]
vocab = Counter(corpus)
# TODO: Remove rare words below frequency threshold
threshold = 5
vocab = {word: count for word, count in vocab.items() if count >= threshold}

sorted_vocab = sorted(vocab.keys())
word2idx = {word: i for i, word in enumerate(sorted_vocab)}
idx2word = {i: word for i, word in enumerate(sorted_vocab)}

# TODO: Filter corpus to known vocabulary and convert to indices


# TODO: Print corpus size and vocabulary size

corpus_idxs = []
corpus_idxs = [[word2idx[word] for word in doc if word in word2idx] for doc in docs]
total_tokens = sum(len(doc_idx) for doc_idx in corpus_idxs)

print(f"Total tokens in filtered corpus: {total_tokens}")
print(f"Vocabulary size (unique words): {len(word2idx)}")

Total tokens in filtered corpus: 513284
Vocabulary size (unique words): 9116


In [17]:
def tf(doc):
    counts = Counter(doc)
    total = len(doc)
    return {word: count / total for word, count in counts.items()}

tf_docs = [tf(doc) for doc in docs]

def idf(docs):
    import math
    from collections import Counter
    N = len(docs)

    # 1. Count how many documents each word appears in (Efficiently)
    # We use set(doc) so we don't count the same word twice in one document
    doc_counts = Counter()
    for doc in docs:
        doc_counts.update(set(doc))

    # 2. Calculate IDF values
    idf_vals = {
        word: math.log(N / count)
        for word, count in doc_counts.items()
    }
    return idf_vals

idf_vals = idf(docs)

def compute_tfidf(tf_docs, idf_vals):
    tfidf_docs = []
    for tf_doc in tf_docs:
        tfidf_doc = {word: val * idf_vals.get(word, 0) for word, val in tf_doc.items()}
        tfidf_docs.append(tfidf_doc)
    return tfidf_docs

tfidf_docs = compute_tfidf(tf_docs, idf_vals)

In [19]:
def top_k_words(tfidf_doc, k=5):
    return sorted(tfidf_doc.items(), key=lambda x: x[1], reverse=True)[:k]

for i, tfidf_doc in enumerate(tfidf_docs):
    print(f"Book {i+1}: {top_k_words(tfidf_doc, 10)}")



Book 1: [('quirrell', 0.0013138893341449332), ('flamel', 0.0008403683578234621), ('quirrells', 0.0007110809181583141), ('nicolas', 0.0005494716185768791), ('turban', 0.0004525060388280181), ('ronan', 0.00043721347835971186), ('chessmen', 0.00040164300401048804), ('griphook', 0.00038786231899544404), ('smelting', 0.00035143762850917707), ('piers', 0.0003279101087697839)]
Book 2: [('lockhart', 0.0024587249704107732), ('gilderoy', 0.0010896506483634274), ('dobby', 0.0010505846185594764), ('myrtle', 0.0008881528934876349), ('lockharts', 0.0007369779328939949), ('heir', 0.0005669061022261499), ('kwikspell', 0.0005207841961876925), ('stall', 0.0005207841961876925), ('diary', 0.0004952756058923246), ('justin', 0.0004493101468770957)]
Book 3: [('lupin', 0.0022445637347849774), ('pettigrew', 0.0013443110378450179), ('marge', 0.0008096418750657495), ('scabbers', 0.0006430371780735341), ('dementors', 0.0006127052357115749), ('crookshanks', 0.0005702405164048322), ('ern', 0.000542077188206776), ('

In [20]:
def tfidf_vector_for_word(word, tfidf_docs):
    return [doc.get(word, 0.0) for doc in tfidf_docs]

word = "harry"
word_dist = tfidf_vector_for_word(word, tfidf_docs)
print(f"Distribution of '{word}': {word_dist}")

Distribution of 'harry': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


### Questions


1.   In the TF-IDF vector you just computed for the word "harry", what does each element of the vector represent?


> Your Answer



2.   In your own words, what are the main limitations of TF-IDF as a word representation? Consider whether TF-IDF captures meaning, similarity between words, and relationships such as analogies.

> Your Answer



## 2. Word2Vec: CBOW and Skip-Gram

TF-IDF represents words using **counts and statistics**, but it does not capture *semantic meaning*.
To go beyond this, we rely on a key idea in linguistics and NLP:

> **You shall know a word by the company it keeps.**

This idea is known as the **distributional hypothesis**.

Instead of counting words, we now **learn** word vectors by training a model to predict words from their surrounding context.

---

## Word2Vec: Learning Word Embeddings

Word2Vec learns *static* (context-independent) word embeddings by solving a **prediction task**.
However, directly predicting a word from the entire vocabulary using a softmax is computationally expensive.

To make training efficient, Word2Vec uses **negative sampling**.

---

## Word2Vec: Two Training Objectives

Word2Vec uses one of two training objectives to generate these positive and negative examples:

### 🔹 CBOW (Continuous Bag of Words)
- **Input:** surrounding context words
- **Target:** the center word

### 🔹 Skip-Gram
- **Input:** center word
- **Target:** surrounding context words
---
### 🔹 What is Negative Sampling?

Negative sampling reformulates word prediction as a **binary classification problem**:

- **Positive examples**: word pairs that actually occur together in the text
- **Negative examples**: randomly sampled word pairs that do *not* occur together

Instead of scoring against the whole vocabulary, the model only compares:
- one true (positive) word
- a small number of randomly sampled (negative) words

This dramatically reduces computation while still producing high-quality embeddings.

### 🔹 Why Negative Sampling Works

By training the model to:
- **increase similarity** for real context–word pairs
- **decrease similarity** for randomly sampled pairs

 the learned vectors capture semantic and syntactic relationships between words.

---

In this section, you will:

1. Generate **(context, target)** training pairs from text  
2. Implement **CBOW and Skip-Gram with negative sampling from scratch**  
3. Train word embeddings using a simple neural objective  
4. Explore the learned embeddings by:
   - Finding nearest neighbors
   - Performing word analogies
   - Comparing CBOW vs Skip-Gram behavior

Do **not** use `gensim` or prebuilt Word2Vec implementations.

The goal is to understand **how embeddings are learned**, not just how to use them.


### Build sliding windows

Use `torch.unfold` to create windows of size `2w+1`, then split into:
- `centers`: middle token  
- `contexts`: left+right tokens (center removed)


In [21]:
def make_windows(corpus_idxs, window_size):
    contexts = []
    centers = []
    for doc in corpus_idxs:
        doc_tensor = torch.tensor(doc)
        if len(doc_tensor) <= 2 * window_size:
            continue
        unfolded = doc_tensor.unfold(0, 2 * window_size + 1, 1)
        for window in unfolded:
            center = window[window_size].item()
            context = torch.cat((window[:window_size], window[window_size+1:])).tolist()
            contexts.append(context)
            centers.append(center)
    return torch.tensor(contexts), torch.tensor(centers)

### Negative-sampling distribution

Build the unigram distribution $P(w) \propto \text{count}(w)^{0.75}$ for sampling negative words.


In [23]:
def build_unigram_dist(word_counts, vocab_size, word2idx, power=0.75):
    dist = torch.zeros(vocab_size)
    for word, idx in word2idx.items():
        dist[idx] = word_counts[word] ** power
    return dist / dist.sum()

unigram_dist = build_unigram_dist(Counter([word for doc in docs for word in doc]), len(word2idx), word2idx)


### Negative sampling utilities

Sample $k$ negative words per example and compute the **negative sampling loss**:  

$$\log\sigma(s^{+}) + \sum \log\sigma(-s^{-})$$


In [24]:

@torch.no_grad()
def sample_negatives(dist_on_device, batch_size, k):
    return torch.multinomial(dist_on_device, batch_size * k, replacement=True).view(batch_size, k)

def negative_sampling_loss(pos_score, neg_score):
    pos_loss = -F.logsigmoid(pos_score).mean()
    neg_loss = -F.logsigmoid(-neg_score).mean()
    return pos_loss + neg_loss

### CBOW
Create a CBOW dataset (`contexts → center`) and a CBOW model that returns positive/negative scores for the **negative sampling loss**.


In [25]:
class CBOWDataset(Dataset):
    def __init__(self, contexts, centers):
        self.contexts = contexts
        self.centers = centers

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        return self.contexts[idx], self.centers[idx]



In [26]:

class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, context_idxs, target_idxs, neg_idxs):
        context_vecs = self.in_embed(context_idxs).mean(dim=1)
        target_vecs = self.out_embed(target_idxs)
        neg_vecs = self.out_embed(neg_idxs)

        pos_score = (context_vecs * target_vecs).sum(dim=1)
        neg_score = torch.bmm(neg_vecs, context_vecs.unsqueeze(2)).squeeze(2)
        return pos_score, neg_score

#### Train CBOW

Training loop for CBOW: sample negatives, compute scores, apply negative sampling loss, and update embeddings.


In [28]:
contexts, centers = make_windows(corpus_idxs, window_size=2)
cbow_dataset = CBOWDataset(contexts, centers)
cbow_loader = DataLoader(cbow_dataset, batch_size=2048, shuffle=True)
cbow = CBOW(len(word2idx), 50).to(device)
cbow_opt = torch.optim.Adam(cbow.parameters(), lr=0.003)
unigram_dist_dev = unigram_dist.to(device)

for epoch in range(5):
    total_loss = 0
    for ctx, center in cbow_loader:
        ctx, center = ctx.to(device), center.to(device)
        neg = sample_negatives(unigram_dist_dev, center.size(0), 10)
        cbow_opt.zero_grad()
        pos_s, neg_s = cbow(ctx, center, neg)
        loss = negative_sampling_loss(pos_s, neg_s)
        loss.backward()
        cbow_opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(cbow_loader)}")


Epoch 1, Loss: 2.7912575049229353


KeyboardInterrupt: 

### Skip-gram

Create (`center`, `context`) training pairs and a Skip-gram model that scores positive and negative contexts for the negative sampling loss.


In [29]:
class SkipGramPairDataset(Dataset):
    def __init__(self, contexts, centers):
        self.pairs = []
        for ctx_list, center in zip(contexts, centers):
            for word in ctx_list:
                self.pairs.append((center, word))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]

In [30]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, centers, pos_ctx, neg_ctx):
        center_vecs = self.in_embed(centers)
        pos_vecs = self.out_embed(pos_ctx)
        neg_vecs = self.out_embed(neg_ctx)

        pos_score = (center_vecs * pos_vecs).sum(dim=1)
        neg_score = torch.bmm(neg_vecs, center_vecs.unsqueeze(2)).squeeze(2)
        return pos_score, neg_score

#### Train Skip-gram

Training loop for Skip-gram with negative sampling: sample negatives, compute scores, apply negative sampling loss, and update embeddings.


In [ ]:
sg_dataset = SkipGramPairDataset(contexts, centers)
sg_loader = DataLoader(sg_dataset, batch_size=4096, shuffle=True)
skipgram = SkipGram(len(word2idx), 50).to(device)
sg_opt = torch.optim.Adam(skipgram.parameters(), lr=0.003)

for epoch in range(5):
    total_loss = 0
    for center, pos in sg_loader:
        center, pos = center.to(device), pos.to(device)
        neg = sample_negatives(unigram_dist_dev, center.size(0), 10)
        sg_opt.zero_grad()
        pos_s, neg_s = skipgram(center, pos, neg)
        loss = negative_sampling_loss(pos_s, neg_s)
        loss.backward()
        sg_opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(sg_loader)}")

### Inspect learned embeddings

Implement helper functions to (1) fetch a word vector from the input/output embedding tables and (2) find nearest neighbors using the combined embedding $(E_{in}+E_{out})/2$.


In [31]:
def get_embedding(model, word, embedding_space="in"):
    idx = torch.tensor([word2idx[word]]).to(device)
    if embedding_space == "in":
        return model.in_embed(idx).detach()
    return model.out_embed(idx).detach()


@torch.no_grad()
def nearest_words_combined(model, word, k=10):
    if word not in word2idx: return []
    w_in = model.in_embed.weight.detach()
    w_out = model.out_embed.weight.detach()
    combined = (w_in + w_out) / 2

    word_vec = combined[word2idx[word]].unsqueeze(0)
    cos_sim = F.cosine_similarity(combined, word_vec)
    vals, idxs = torch.topk(cos_sim, k + 1)
    return [idx2word[i.item()] for i in idxs[1:]]

### Query nearest neighbors

Compare CBOW vs Skip-gram by printing the top-5 nearest words to a query term **"hogwarts"** using the combined embedding space.


In [32]:
word = "hogwarts"
print(f"CBOW | Nearest words for '{word}': {nearest_words_combined(cbow, word, 5)}")
print(f"Skip-gram | Nearest words for '{word}': {nearest_words_combined(skipgram, word, 5)}")


CBOW | Nearest words for 'hogwarts': ['cluster', 'stricken', 'tournament', 'valuable', 'confirmed']


NameError: name 'skipgram' is not defined

## Questions

You observed the nearest neighbors of **"hogwarts"** using both CBOW and Skip-Gram (You are encouraged to test additional words to build better intuition).


Answer the following questions:  


1. **Which model produced more semantically meaningful neighbors?**  
   Justify your answer based on the words returned.

   > Your Answer

2. **Why do you think CBOW shows higher cosine similarity scores than Skip-Gram?**  
   Consider how each model aggregates or distributes context information.

   > Your Answer

3. **Why might Skip-Gram return more diverse or less “obvious” neighbors?**  
   What does this tell you about how Skip-Gram learns word representations?
   > Your Answer



## Part 2: GloVe – Global Vectors for Word Representation

GloVe is a pre-trained word embedding model developed by Stanford University. It learns word representations by leveraging global word–word co-occurrence statistics from large text corpora, resulting in embeddings that capture semantic relationships between words. Words with similar meanings tend to have vectors that are close to each other in the embedding space.

At the core of GloVe is the construction of a **global co-occurrence matrix** $X$, where each entry $X_{ij}$ represents how often word $j$ appears in the context of word $i$ across the entire corpus. Rather than using these raw counts directly, GloVe learns word embeddings by minimizing the following weighted least-squares objective:

$$
J = \sum_{i,j=1}^{V} f(X_{ij})
\left( \mathbf{w}_i^\top \mathbf{\tilde{w}}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2
$$

where:
- $\mathbf{w}_i$ is the vector representation of the target word $i$
- $\mathbf{\tilde{w}}_j$ is the vector representation of the context word $j$
- $b_i$ and $\tilde{b}_j$ are bias terms
- $f(X_{ij})$ is a weighting function that downweights very frequent and very rare co-occurrences

The weighting function is defined as:

$$
f(x) =
\begin{cases}
\left( \frac{x}{x_{\text{max}}} \right)^\alpha & \text{if } x < x_{\text{max}} \\
1 & \text{if } x \ge x_{\text{max}}
\end{cases}
$$

with typical values $x_{\text{max}} = 100$ and $\alpha = 0.75$.

This objective encourages the **dot product of word vectors** to approximate the logarithm of their co-occurrence counts, allowing linear relationships in the embedding space to encode meaningful semantic regularities.

In this implementation, we load a pre-trained GloVe model using **Gensim’s downloader API**. Specifically, the model `glove-wiki-gigaword-300` is used, which was trained on a combination of Wikipedia and Gigaword text containing approximately 6 billion tokens. Each word in the vocabulary is represented by a **300-dimensional** dense vector.

The vocabulary consists of all tokens for which embeddings are available in the pre-trained model. These embeddings can be directly used for downstream NLP tasks such as similarity computation, clustering, or as input features for machine learning models.


In [ ]:
# run this code to install gensim on your environment
!pip install gensim

In [ ]:
from gensim.models import KeyedVectors
import gensim.downloader as api

# TODO: Download and load 300-dimensional GloVe embeddings
glove_name = ...
glove = ...
print("Loaded:", glove_name, " | vocab:", len(glove), " | dim:", glove.vector_size)

### Building GloVe Embedding Utilities

This section constructs helper data structures and functions for working with GloVe embeddings in PyTorch.

First, we build mappings between words and indices (`word2idx` and `idx2word`) and create an embedding matrix `E_glove` of shape `(V, D)`, where `V` is the vocabulary size and `D` is the embedding dimension.

To enable efficient cosine similarity computation, the embedding matrix is L2-normalized once in advance.

Two utility functions are provided:
- **`get_glove_embedding(word)`**: Returns the embedding vector for a given word.
- **`nearest_words_glove(word, k)`**: Finds the top-`k` most similar words to a query word using cosine similarity.

These utilities support fast embedding lookup and semantic similarity queries.


In [ ]:
# TODO: Build word ↔ index mappings and embedding matrix from GloVe

# TODO: GloVe embedding matrix: (vocab_size, embedding_dim)

# TODO: L2-normalize embeddings once for efficient cosine similarity

def get_glove_embedding(word: str):
    # TODO: Retrieve the raw GloVe embedding for a given word
    pass

@torch.no_grad()
def nearest_words_glove(word: str, k: int = 10):
    # TODO: Find k nearest neighbors using cosine similarity


    # TODO: Filter out the query word from results



Test the GloVe similarity function on Harry Potter related words and compare the results with Word2Vec models

In [ ]:
for w in ["harry", "wizard", "magic", "hogwarts", "dumbledore"]:
    try:
        print(w, "->", nearest_words_glove(w, k=5))
    except KeyError as e:
        print("OOV:", e)

### Word Analogies

One standard method for assessing word embeddings, first popularized in *Efficient Estimation of Word Representations in Vector Space*, is to examine pairs of words that exhibit the same semantic or morphological relationship. For example, the pairs *fast → faster* and *slow → slower* follow an identical pattern.

Each pair can be interpreted as an analogy question:  
“What word corresponds to *slow* in the same way that *faster* corresponds to *fast*?”

These relationships can be resolved through vector arithmetic in the embedding space. We compute

$$
x = w_{\text{faster}} - w_{\text{fast}} + w_{\text{slow}}
$$

and then search the vocabulary for the word vector that is most similar to \(x\). In this notebook, similarity is measured using cosine similarity, resulting in the following objective:

$$
w^{*} = \arg\max_{w} \cos\left(w,\; w_{\text{faster}} - w_{\text{fast}} + w_{\text{slow}}\right)
$$


In [ ]:
@torch.no_grad()
def vector_arithmetic(model, a, b, c, top_k=5):
    # TODO: Compute analogy vector: b - a + c

    # TODO: Cosine similarity via dot product

    # TODO: Exclude input words


In [ ]:
# TODO: Verify that vector arithmetic produces the expected analogy result using GloVe vectors
test_set = [
    ["rome", "italy", "madrid", "spain"],
    ["berlin", "germany", "vienna", "austria"],
    ["beijing", "china", "seoul", "korea"],
    ["king", "queen", "prince", "princess"],
    ["actor", "actress", "waiter", "waitress"],
    ["teach", "teacher", "drive", "driver"],
    ["happy", "happier", "cold", "colder"],
    ["strong", "strongest", "weak", "weakest"],
    ["write", "writing", "swim", "swimming"],
    ["buy", "bought", "go", "went"],
    ["leaf", "leaves", "wolf", "wolves"],
    ["child", "children", "person", "people"],
    ["spain", "spanish", "greece", "greek"],
]




# Section 2: Geometry of Fine-Tuning — Measuring ΔM and ΔD During Training

## Context
In recent papers on **parameter-efficient fine-tuning (PEFT)**, authors analyze *how weights move* during training. Two common geometry-inspired quantities are:

- **ΔM (magnitude change)**: how much *column norms* change.
- **ΔD (direction change)**: how much *column directions* change, often via cosine similarity.

In this homework, you will replicate a lightweight version of a "paper-style" analysis on a small BERT model, comparing:
1. **Full fine-tuning**
2. **LoRA(https://arxiv.org/abs/2106.09685)**
3. **DoRA(https://arxiv.org/abs/2402.09353)**

You will:
- fine-tune on **AG News (4-class)**,
- capture intermediate weight snapshots for **attention query/value** projection matrices,
- compute ΔM and ΔD over time,
- visualize the relationship between ΔM and ΔD,
- and answer analysis questions about what you observe.

## Learning Objectives
By the end, you should be able to:
- Implement a reproducible NLP training pipeline with `datasets` and `transformers`.
- Implement *column-wise* geometry metrics for weight updates.
- Instrument training to capture intermediate model states.
- Compare training-time geometry across optimization regimes (Full FT vs PEFT).
- Communicate empirical findings with appropriate plots and statistical summaries.

## What you must submit
1. This notebook with all **@ToDo** items completed.
2. All plots requested in the notebook.
3. Short written answers to the **analysis questions** at the end.

## Rules
- Do not change the definitions of ΔM and ΔD beyond what is specified (unless explicitly asked).
- You may tune hyperparameters *only* where the notebook indicates it is allowed.
- Your analysis must be based on your results (not copied from others).


---
## Step 0 — Environment and Setup

### What you should do and why
You will install the required libraries and set deterministic seeds. Reproducibility is essential for comparing geometry across methods: if runs are not controlled, you may attribute differences to LoRA/DoRA when they are actually caused by randomness.


In [ ]:

# ============================================================
# LIGHT setup that works with datasets==4.0.0:
# Model: google/bert_uncased_L-4_H-256_A-4  (Tiny BERT ~14M params)
# Dataset: ag_news (4-class classification; no dataset script)
# Methods: Full FT vs LoRA vs DoRA (if supported)
# Paper-style geometry: ΔM / ΔD on attention QUERY/VALUE weights, column-wise
# Compute points across intermediate steps (like paper) and plot:
#   - Scatter ΔD vs ΔM (per method)
#   - Density-like (hexbin) optional
#   - Correlation table (Pearson/Spearman)
# ============================================================

# If needed (uncomment):
# !pip -q install transformers datasets evaluate accelerate peft

import os, re, time, math, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
import evaluate
import peft
from peft import LoraConfig, get_peft_model

# -------------------------
# 0.1) Reproducibility / device
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

import transformers, datasets
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)



---
## Step 1 — Load the Dataset (AG News)

### What you should do and why
You will load **AG News** and create smaller train/eval splits to keep the homework runnable on modest hardware. The goal is not to maximize accuracy, but to **compare geometry across training methods under similar conditions**.


In [ ]:

# -------------------------
# 1) Dataset: AG News (script-free)
# -------------------------
ds = load_dataset("ag_news")

TRAIN_SIZE = 8000 if device == "cuda" else 1500
EVAL_SIZE  = 1500 if device == "cuda" else 400

TRAIN_SIZE = min(TRAIN_SIZE, len(ds["train"]))
EVAL_SIZE  = min(EVAL_SIZE, len(ds["test"]))

train_ds = ds["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
eval_ds  = ds["test"].shuffle(seed=SEED).select(range(EVAL_SIZE))
print("train size:", len(train_ds), "eval size:", len(eval_ds))

# Inspect a few examples (sanity check)
for i in range(2):
    print(f"\nExample {i}:")
    print("label:", train_ds[i]["label"])
    print("text :", train_ds[i]["text"][:200], "…")



---
## Step 2 — Tokenization and Dataset Preprocessing

### What you should do and why
Transformers expect tokenized inputs. You will implement a batching tokenizer function that:
- tokenizes text with truncation,
- sets a maximum length,
- and creates a `labels` field required by `Trainer`.

This step is foundational: incorrect labels or tokenization will invalidate the training and the geometry analysis.


In [ ]:

# -------------------------
# 2) Model + Tokenizer
# -------------------------
model_name = "google/bert_uncased_L-4_H-256_A-4"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_with_labels(batch):
    """
    Tokenize a batch of examples and attach labels.

    Expected output keys (at least):
      - input_ids
      - attention_mask
      - labels

    @ToDo:
    - Use `tokenizer(batch["text"], truncation=True, max_length=192)`.
    - Copy `batch["label"]` into output["labels"].
    - Return the output dict.
    """
    # @ToDo
    raise NotImplementedError("Implement tokenize_with_labels")

# Apply tokenization
train_tok = train_ds.map(tokenize_with_labels, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(tokenize_with_labels,  batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tok[0].keys())



---
## Step 3 — Define Evaluation Metrics

### What you should do and why
Accuracy is a simple sanity metric for AG News. You will implement the `compute_metrics` callback that converts logits into predicted class IDs.

This is used to:
- compare Full FT vs LoRA vs DoRA in terms of *task performance*,
- and ensure training is actually working before interpreting geometry plots.


In [ ]:

# -------------------------
# 3) Metrics
# -------------------------
acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """
    eval_pred is (logits, labels).
    Return a dict like {"accuracy": value}.

    @ToDo:
    - Extract logits and labels.
    - Compute predictions with argmax over the last axis.
    - Return accuracy via `acc.compute(predictions=..., references=...)`.
    """
    # @ToDo
    raise NotImplementedError("Implement compute_metrics")



---
## Step 4 — Training Helpers (Parameters, Arguments)

### What you should do and why
You will implement lightweight helper functions:
- count parameters (total vs trainable),
- create TrainingArguments compatible with older/newer Transformers.

These utilities help quantify the *efficiency* of PEFT methods and keep the training loop clean.


In [ ]:

# -------------------------
# 4) Training helpers
# -------------------------
def count_params(model):
    """
    Return (total_params, trainable_params).

    @ToDo:
    - total_params: sum of p.numel() for all parameters
    - trainable_params: sum of p.numel() for parameters where p.requires_grad
    """
    # @ToDo
    raise NotImplementedError("Implement count_params")

def short_model_report(model, title="model"):
    total, trainable = count_params(model)
    print(f"[{title}] total params: {total:,}")
    print(f"[{title}] trainable params: {trainable:,} ({trainable/total:.4%})")

def make_training_args(outdir, lr, epochs, batch=32, eval_every="epoch"):
    """
    Create TrainingArguments with backward-compatible naming.

    Note: transformers changed `evaluation_strategy` -> `eval_strategy` in some versions.
    """
    base = dict(
        output_dir=outdir,
        report_to=[],
        logging_steps=50,
        save_strategy="no",
        fp16=(device=="cuda"),
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="cosine",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch if device=="cuda" else max(8, batch//2),
        per_device_eval_batch_size=64,
        gradient_accumulation_steps=1,
    )
    try:
        return TrainingArguments(evaluation_strategy=eval_every, **base)
    except TypeError:
        return TrainingArguments(eval_strategy=eval_every, **base)



---
## Step 5 — Implement Paper-Style ΔM and ΔD (Column-Wise)

### What you should do and why
You will implement the *core geometry metrics*:

- Let W ∈ R^{d_out × d_in}.
- Consider each **column** as a vector in R^{d_out}.

Definitions (column-wise):
- Column norms: ||W[:, j]||_2
- **ΔM**: mean absolute change in column norms from initialization
- **ΔD**: 1 − mean cosine similarity between current and initial columns

These are used in the final scatter plot of ΔD vs ΔM and should match the definitions in the provided design.


In [ ]:

# -------------------------
# 5) Paper-style ΔM / ΔD definitions (column-wise)
# -------------------------
def col_norms(W: torch.Tensor) -> torch.Tensor:
    """
    Return a vector of L2 norms for each column of W.

    @ToDo:
    - Use torch.linalg.vector_norm with dim=0.
    - Cast to float for stability.
    """
    # @ToDo
    raise NotImplementedError("Implement col_norms")

def mean_cosine_cols(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """
    Compute mean cosine similarity between corresponding columns of A and B.

    @ToDo:
    - Normalize columns of A and B (clamp norms to avoid divide-by-zero).
    - Compute cosine for each column: sum(An * Bn) over rows.
    - Return the mean cosine over columns.
    """
    # @ToDo
    raise NotImplementedError("Implement mean_cosine_cols")

def delta_M_D(W0: torch.Tensor, Wt: torch.Tensor):
    """
    Compute (ΔM, ΔD) between initial weight W0 and current weight Wt.

    Definitions:
      m0 = column norms of W0
      mt = column norms of Wt
      ΔM = mean |mt - m0|
      cos_mean = mean cosine similarity between columns of Wt and W0
      ΔD = 1 - cos_mean

    @ToDo:
    - Implement using col_norms and mean_cosine_cols.
    - Return Python floats (not tensors).
    """
    # @ToDo
    raise NotImplementedError("Implement delta_M_D")



---
## Step 6 — Capture Intermediate Weight Snapshots (Query/Value Only)

### What you should do and why
We need intermediate snapshots across training (not only at the end). You will:

1. Identify **target parameter names** matching BERT attention query/value weights:
   - `bert.encoder.layer.{i}.attention.self.query.weight`
   - `bert.encoder.layer.{i}.attention.self.value.weight`

2. Implement a TrainerCallback that captures these tensors at selected training steps.

This instrumentation enables time-resolved geometry analysis similar to paper plots showing how weights evolve *during* fine-tuning.


In [ ]:

# -------------------------
# 6) Capture intermediate snapshots (only query/value weights)
# -------------------------
TARGET_REGEX = re.compile(r"bert\.encoder\.layer\.\d+\.attention\.self\.(query|value)\.weight$")

def select_target_param_names(model):
    """
    Return sorted parameter names that match TARGET_REGEX.

    @ToDo:
    - Iterate model.named_parameters()
    - Select names that match TARGET_REGEX
    - Return sorted list of names
    """
    # @ToDo
    raise NotImplementedError("Implement select_target_param_names")

class WeightSnapshotCallback(TrainerCallback):
    """Captures a CPU clone of selected parameters at specified training steps."""
    def __init__(self, param_names, steps_to_capture):
        self.param_names = set(param_names)
        self.steps_to_capture = set(steps_to_capture)
        self.snapshots = {}  # step -> {name: tensor_cpu}

    def on_step_end(self, args, state, control, **kwargs):
        """
        @ToDo:
        - If current step is in steps_to_capture and not already captured:
          - Iterate model.named_parameters()
          - If name is in self.param_names: store p.detach().cpu().clone()
          - Save under self.snapshots[step]
        """
        # @ToDo
        raise NotImplementedError("Implement on_step_end snapshot capture")

def choose_capture_steps(total_steps):
    """
    Choose capture steps similar to paper-style intermediate points.

    @ToDo:
    - Return a sorted unique list containing:
      [0, 20%, 50%, 80%, 100%] of total_steps (rounded to int)
    - Ensure steps are at least 1 where appropriate (except 0).
    """
    # @ToDo
    raise NotImplementedError("Implement choose_capture_steps")



---
## Step 7 — Training With Snapshots

### What you should do and why
You will write a training wrapper that:
- creates a callback with selected capture steps,
- stores an explicit step-0 snapshot (initial weights),
- trains with `Trainer.train()`,
- evaluates at the end,
- and returns snapshots + metrics.

This wrapper ensures the only difference between methods is **which parameters are trained**, not how we collect geometry.


In [ ]:

# -------------------------
# 7) Training loop with snapshots
# -------------------------
def train_with_snapshots(model, outdir, lr, epochs, batch=32):
    """
    Train a model while capturing intermediate snapshots of query/value weights.

    Returns:
      trainer, metrics_dict, snapshots_dict

    @ToDo:
    - Move model to device.
    - Select target param names with select_target_param_names.
    - Estimate total_steps and choose capture steps.
      Hint: steps_per_epoch = ceil(len(train_tok) / per_device_batch_size)
    - Instantiate WeightSnapshotCallback.
    - Create TrainingArguments via make_training_args.
    - Create Trainer with callbacks=[cb].
    - Store an explicit snapshot at step=0 in cb.snapshots.
    - Train, measure train time, evaluate.
    - Return (trainer, metrics, cb.snapshots).
    """
    # @ToDo
    raise NotImplementedError("Implement train_with_snapshots")



---
## Step 8 — Build ΔM/ΔD Point Cloud

### What you should do and why
After training, you will convert snapshots into a table where each row corresponds to:
- a method (full/lora/dora),
- a training step,
- a specific weight matrix name,
- and its geometry relative to initialization.

This tabular representation supports scatter plots, density plots, and correlation analysis.


In [ ]:

# -------------------------
# 8) Build ΔM/ΔD points
# -------------------------
def build_points(snapshots, method):
    """
    Create a DataFrame with columns:
      method, step, name, dM, dD

    For each captured step and each target weight matrix:
      compute (ΔM, ΔD) between W0 and Wt.

    @ToDo:
    - Sort steps.
    - Let W0 = snapshots[0].
    - For each step and each name present in W0:
        - compute dM, dD = delta_M_D(W0[name], snapshots[step][name])
        - append row
    - Return pd.DataFrame(rows)
    """
    # @ToDo
    raise NotImplementedError("Implement build_points")



---
## Step 9 — Correlation Utilities (Pearson and Spearman)

### What you should do and why
A typical paper-style question is: *are magnitude and direction changes related?*

You will implement:
- Pearson correlation (linear association),
- Spearman correlation (rank-based, monotonic association).

These will be computed per method on the point cloud.


In [ ]:

# -------------------------
# 9) Correlation helpers
# -------------------------
def pearson(x, y):
    """
    @ToDo:
    - Convert to float arrays.
    - Drop non-finite values.
    - Return np.corrcoef(x, y)[0,1] if length >= 3 else nan.
    """
    # @ToDo
    raise NotImplementedError("Implement pearson")

def spearman(x, y):
    """
    @ToDo:
    - Convert to float arrays.
    - Drop non-finite values.
    - Rank x and y (average ranks for ties).
    - Return Pearson correlation of ranks if length >= 3 else nan.
    """
    # @ToDo
    raise NotImplementedError("Implement spearman")



---
## Step 10 — Define Methods: Full FT vs LoRA vs DoRA

### What you should do and why
You will instantiate and train three variants:

1. **Full fine-tuning**: all parameters trainable.
2. **LoRA**: train low-rank adapters on attention query/value modules.
3. **DoRA**: if supported, uses a decomposition that can alter geometry differently than LoRA.

You will record:
- number of trainable parameters,
- final eval accuracy,
- training time,
- and geometry point clouds.


In [ ]:

# -------------------------
# 10) Hyperparameters (adjust ONLY if runtime requires)
# -------------------------
EPOCHS_FULL = 3
EPOCHS_ADAPTER = 6
LR_FULL = 2e-5
LR_ADAPTER = 3e-4
BATCH = 32

def make_lora_model(model_name, num_labels=4, use_dora=False):
    """
    Create a (LoRA or DoRA) model for sequence classification.

    @ToDo:
    - Load base model: AutoModelForSequenceClassification.from_pretrained(...)
    - Create LoraConfig with:
        r=8, lora_alpha=16, lora_dropout=0.05,
        bias="none", task_type="SEQ_CLS",
        target_modules=["query", "value"]
      If use_dora=True, include use_dora=True in config (if supported).
    - Wrap with get_peft_model and return it.
    """
    # @ToDo
    raise NotImplementedError("Implement make_lora_model")


In [ ]:

# -------------------------
# 10.1) Train: Full FT
# -------------------------
full_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
short_model_report(full_model, "full-ft (before)")

trainer_full, metrics_full, snaps_full = train_with_snapshots(
    full_model,
    outdir="out_full_agnews_bert4",
    lr=LR_FULL,
    epochs=EPOCHS_FULL,
    batch=BATCH
)
print("full metrics:", metrics_full)
print("full captured steps:", sorted(snaps_full.keys()))


In [ ]:

# -------------------------
# 10.2) Train: LoRA
# -------------------------
lora_model = make_lora_model(model_name, num_labels=4, use_dora=False)
short_model_report(lora_model, "lora (before)")
try:
    lora_model.print_trainable_parameters()
except Exception as e:
    print("Could not print trainable parameters:", e)

trainer_lora, metrics_lora, snaps_lora = train_with_snapshots(
    lora_model,
    outdir="out_lora_agnews_bert4",
    lr=LR_ADAPTER,
    epochs=EPOCHS_ADAPTER,
    batch=BATCH
)
print("lora metrics:", metrics_lora)
print("lora captured steps:", sorted(snaps_lora.keys()))


In [ ]:

# -------------------------
# 10.3) Train: DoRA (if supported)
# -------------------------
snaps_dora = None
metrics_dora = {"skipped": True}

try:
    dora_model = make_lora_model(model_name, num_labels=4, use_dora=True)
    short_model_report(dora_model, "dora (before)")
    try:
        dora_model.print_trainable_parameters()
    except Exception as e:
        print("Could not print trainable parameters:", e)

    trainer_dora, metrics_dora, snaps_dora = train_with_snapshots(
        dora_model,
        outdir="out_dora_agnews_bert4",
        lr=LR_ADAPTER,
        epochs=EPOCHS_ADAPTER,
        batch=BATCH
    )
    print("dora metrics:", metrics_dora)
    print("dora captured steps:", sorted(snaps_dora.keys()))

except TypeError as e:
    print("DoRA not supported by this PEFT version. Skipping DoRA.")
    print("Error:", e)



---
## Step 11 — Assemble All Points and Compute Correlations

### What you should do and why
You will build a combined DataFrame for all methods and focus on **query weights** (often closest to paper plots).

Then you will compute correlation summaries:
- Pearson(ΔD, ΔM)
- Spearman(ΔD, ΔM)

These summaries help quantify trends you see in scatter plots.


In [ ]:

# -------------------------
# 11) Build ΔM/ΔD points + correlations
# -------------------------
df_full = build_points(snaps_full, "full")
df_lora = build_points(snaps_lora, "lora")

dfs = [df_full, df_lora]
if snaps_dora is not None:
    df_dora = build_points(snaps_dora, "dora")
    dfs.append(df_dora)

df = pd.concat(dfs, ignore_index=True)
print("points:", len(df))

# Focus on QUERY only (often matches paper plots better)
df_q = df[df["name"].str.contains(r"\.query\.weight$", regex=True)].copy()
print("points (query only):", len(df_q))

summary = []
for m in sorted(df_q["method"].unique()):
    sub = df_q[df_q["method"] == m]
    summary.append({
        "method": m,
        "n_points": len(sub),
        "pearson(dD, dM)": pearson(sub["dD"], sub["dM"]),
        "spearman(dD, dM)": spearman(sub["dD"], sub["dM"]),
    })
summary_df = pd.DataFrame(summary)
summary_df



---
## Step 12 — Plots: Scatter and Hexbin (Paper-Style)

### What you should do and why
You will plot ΔD vs ΔM for each method.
- Scatter plots show raw point clouds.
- Hexbin plots provide a density-like view similar to many papers.

Include Pearson and Spearman in the plot titles to connect visual patterns with quantitative summaries.


In [ ]:

def scatter_all(df_use, title_suffix="(query weights)"):
    """
    For each method in df_use:
      - create a scatter plot of ΔD vs ΔM
      - compute Pearson/Spearman for that method
      - include them in the title

    @ToDo:
    - Iterate over methods.
    - Extract x=ΔD and y=ΔM, filter finite values.
    - Compute r=pearson(x,y), rs=spearman(x,y).
    - Create a matplotlib scatter plot with labels and title.
    """
    # @ToDo
    raise NotImplementedError("Implement scatter_all")

def hexbin_all(df_use, gridsize=35, title_suffix="(query weights)"):
    """
    For each method in df_use:
      - create a hexbin plot of ΔD vs ΔM
      - compute Pearson/Spearman and include them in the title
      - include a colorbar (count)

    @ToDo:
    - Similar to scatter_all but use plt.hexbin and plt.colorbar.
    """
    # @ToDo
    raise NotImplementedError("Implement hexbin_all")


In [ ]:

# Produce plots (once you implement plotting functions)
scatter_all(df_q)
hexbin_all(df_q)



---
## Step 13 — Training Metrics Summary Table

### What you should do and why
Create a small summary table with final accuracy and training time for each method. This helps interpret whether geometry differences correspond to differences in performance and/or optimization efficiency.


In [ ]:

train_results = [
    {"method": "full", "eval_accuracy": metrics_full.get("eval_accuracy"), "train_seconds": metrics_full.get("train_seconds")},
    {"method": "lora", "eval_accuracy": metrics_lora.get("eval_accuracy"), "train_seconds": metrics_lora.get("train_seconds")},
]
if snaps_dora is None:
    train_results.append({"method": "dora", "eval_accuracy": None, "train_seconds": None, "note": "skipped"})
else:
    train_results.append({"method": "dora", "eval_accuracy": metrics_dora.get("eval_accuracy"), "train_seconds": metrics_dora.get("train_seconds")})

pd.DataFrame(train_results)



---
# Analysis Questions (Answer in Markdown)

Answer concisely but precisely. Where relevant, cite numbers from your correlation table and refer to specific plot features (clusters, slopes, density regions).

1. **Sanity checks**
   1. What accuracy did you obtain for each method?
   2. Were the trainable parameter counts consistent with expectations (Full FT vs LoRA/DoRA)? Provide the counts.

2. **Geometry–performance relationship**
   1. Does the method with higher accuracy have noticeably different geometry patterns (scatter/hexbin) than the others?
   2. Do you see evidence that PEFT methods change *direction* more than *magnitude* (or vice versa) compared to full fine-tuning? Explain.

3. **Correlation interpretation**
   1. Compare Pearson vs Spearman for each method. Are they similar? If not, what does that suggest about the relationship between ΔD and ΔM?
   2. Which method has the strongest association between ΔD and ΔM? Provide correlation values.

4. **Step-wise dynamics**
   1. Look at points grouped by step (you may color points by step if you want). Do early steps differ from late steps?
   2. Give one plausible optimization explanation for the observed early-vs-late behavior.

5. **Query vs Value**
   1. Repeat the analysis for **value weights** only (`.value.weight`). Do conclusions change?
   2. Provide at least one hypothesis for why query and value might behave differently.

6. **Method critique**
   1. Identify at least two limitations of this experiment design (e.g., dataset size, step sampling, metric choices).
   2. Propose one extension that would strengthen the evidence (e.g., multiple seeds, additional layers, other target modules).

7. **Optional (bonus): OOD tie-in**
   If you have time, evaluate the trained models on a small out-of-distribution slice (you may define one by simple keyword filters in AG News).
   1. Do geometry patterns correlate with OOD robustness in your quick test?
   2. What additional experiments would be needed to make a credible claim?
